<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/02_Llama_3_1_%2B_Ollama_%2B_Llama_Stack_%2B_Guards_%2B_FastAPI_%2B_WhatsApp_Cloud_API_%2B_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETAPA 1 — Instalación y configuración

En esta etapa se prepara el entorno del agente conversacional.

Se instala:

- Ollama
- Llama 3.1 8B
- FastAPI
- Uvicorn
- ngrok
- Sentence Transformers
- FAISS
- Requests

También se cargan las credenciales necesarias para WhatsApp Cloud API y ngrok.

In [2]:
# ============================================================
# ETAPA 1 - INSTALACIÓN Y CONFIGURACIÓN
# ============================================================

!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    sentence-transformers \
    faiss-cpu \
    requests

from pathlib import Path
from google.colab import userdata
import subprocess
import time
import requests

# ------------------------------------------------------------
# RUTAS
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/Mx.Oscar.Hdez")

RAG_DIR = BASE_DIR / "rag"

CHUNKS_FILE = RAG_DIR / "chunks.json"
INDEX_FILE = RAG_DIR / "faiss.index"

# ------------------------------------------------------------
# MODELO
# ------------------------------------------------------------

LLM_MODEL = "llama3.1:8b"
EMBED_MODEL = "intfloat/multilingual-e5-small"

# ------------------------------------------------------------
# CREDENCIALES
# ------------------------------------------------------------

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
WHATSAPP_TOKEN = userdata.get("WHATSAPP_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")

# ------------------------------------------------------------
# INICIAR OLLAMA
# ------------------------------------------------------------

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

# Descargar modelo
!ollama pull llama3.1:8b

# ------------------------------------------------------------
# VERIFICACIÓN
# ------------------------------------------------------------

respuesta = requests.get(
    "http://127.0.0.1:11434/api/tags"
)

print("=" * 60)
print("CONFIGURACIÓN DEL AGENTE")
print("=" * 60)

print("Ollama:", respuesta.status_code == 200)
print("Modelo:", LLM_MODEL)

print("RAG:", CHUNKS_FILE.exists() and INDEX_FILE.exists())

print("NGROK:", bool(NGROK_TOKEN))
print("WHATSAPP_TOKEN:", bool(WHATSAPP_TOKEN))
print("PHONE_NUMBER_ID:", bool(PHONE_NUMBER_ID))
print("VERIFY_TOKEN:", bool(VERIFY_TOKEN))

print()
print("✓ Entorno preparado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

# ETAPA 2 — Carga y consulta del RAG

En esta etapa se reutiliza la base documental creada en el Notebook 1.

Se cargan:

- `chunks.json`
- `faiss.index`
- modelo de embeddings E5

Después se crea una función de recuperación que convierte una pregunta en un embedding y busca los fragmentos más similares en FAISS.

In [3]:
# ============================================================
# ETAPA 2 - CARGA Y CONSULTA DEL RAG
# ============================================================

import json
import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# DISPOSITIVO
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# CARGAR CHUNKS E ÍNDICE
# ------------------------------------------------------------

with open(
    CHUNKS_FILE,
    "r",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

index = faiss.read_index(
    str(INDEX_FILE)
)

# ------------------------------------------------------------
# CARGAR E5
# ------------------------------------------------------------

print("Cargando E5...")

embedding_model = SentenceTransformer(
    EMBED_MODEL,
    device=DEVICE
)

embedding_model.max_seq_length = 512

print("✓ E5 listo en", DEVICE)

# ------------------------------------------------------------
# FUNCIÓN DE RECUPERACIÓN
# ------------------------------------------------------------

TOP_K = 4

def recuperar_contexto(pregunta, top_k=TOP_K):

    vector = embedding_model.encode(
        ["query: " + pregunta],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    vector = np.asarray(
        vector,
        dtype="float32"
    )

    scores, indices = index.search(
        vector,
        top_k
    )

    resultados = []

    for score, posicion in zip(
        scores[0],
        indices[0]
    ):

        chunk = chunks[posicion].copy()
        chunk["score"] = float(score)

        resultados.append(chunk)

    return resultados


print()
print("✓ RAG cargado")
print("Chunks:", len(chunks))
print("Vectores FAISS:", index.ntotal)

Cargando E5...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✓ E5 listo en cuda

✓ RAG cargado
Chunks: 574
Vectores FAISS: 574


In [4]:
pregunta = (
    "¿Cuál es el perfil de egreso "
    "de Ingeniería en Sistemas Computacionales?"
)

resultados = recuperar_contexto(
    pregunta
)

for n, r in enumerate(
    resultados,
    start=1
):
    print()
    print(
        f"[{n}] {r['documento']} "
        f"| pág. {r['pagina']} "
        f"| score {r['score']:.4f}"
    )
    print(r["texto"])


[1] Perfil-Objetivo_ISC.pdf | pág. 1 | score 0.8601
©TecNM mayo 2016 Página | 1 TECNOLÓGICO NACIONAL DE MÉXICO Secretaría Académica, de Investigación e Innovación Dirección de Docencia e Innovación Educativa Ingeniería en Sistemas Computacionales ISIC-2010-224 Objetivo General Formar profesionistas líderes con visión estratégica y amplio sentido ético; capaz de diseñar, desarrollar, implementar y administrar tecnología computacional para aportar soluciones innovadoras en beneficio de la sociedad; en un contexto global, multidisciplinario y sostenible. Perfil de Egreso 1. Implementa aplicaciones computacionales para solucionar problemas de diversos contextos, integrando diferentes tecnologías, plataformas o dispositivos. 2. Diseña, desarrolla y aplica modelos computacionales para solucionar problemas, mediante la selección y uso de herramientas matemáticas. 3. Diseña e implementa interfaces para la automatización de sistemas de hardware y desarrollo del software asociado. 4. Coordina y

# ETAPA 3 — RAG + Llama 3.1

En esta etapa se:

- recuperan los fragmentos más relevantes con FAISS;
- construye un contexto con esos fragmentos;
- envía el contexto y la pregunta a `llama3.1:8b` mediante Ollama;
- genera una respuesta sustentada en el RAG.

El modelo debe responder únicamente con la información recuperada.

In [42]:
# ============================================================
# ETAPA 3 - RAG + LLAMA 3.1 EN OLLAMA
# ============================================================

import requests
import time

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"


def responder_rag(pregunta):

    resultados = recuperar_contexto(
        pregunta
    )

    contexto = "\n\n".join(
        [
            f"""
DOCUMENTO: {r['documento']}
PÁGINA: {r['pagina']}
{r['texto']}
""".strip()
            for r in resultados
        ]
    )

    mensaje = f"""
Eres un asistente académico del Tecnológico Nacional de México.

Debes responder siempre en español.

Utiliza únicamente la información disponible en el CONTEXTO.

Analiza la intención de la pregunta y relaciona conceptos aunque el usuario
no utilice exactamente las mismas palabras que aparecen en los documentos.

Si la respuesta está distribuida en varios fragmentos, integra la información.

Responde de forma clara, breve y directa.

No inventes datos ni completes información que no esté respaldada por el contexto.

Si realmente no existe información suficiente en el contexto, indícalo claramente.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}
""".strip()

    inicio = time.time()

    respuesta = requests.post(
        OLLAMA_URL,
        json={
            "model": LLM_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": mensaje
                }
            ],
            "stream": False
        },
        timeout=180
    )

    respuesta.raise_for_status()

    tiempo = time.time() - inicio

    texto = respuesta.json()[
        "message"
    ]["content"]

    return {
        "respuesta": texto,
        "fuentes": resultados,
        "tiempo": round(tiempo, 2)
    }


print("✓ RAG conectado con Llama 3.1")

✓ RAG conectado con Llama 3.1


In [43]:
pregunta = (
    "¿Cuál es el perfil de egreso "
    "de Ingeniería en Sistemas Computacionales?"
)

resultado = responder_rag(
    pregunta
)

print("=" * 60)
print("RESPUESTA")
print("=" * 60)

print(
    resultado["respuesta"]
)

print()
print(
    "Tiempo:",
    resultado["tiempo"],
    "segundos"
)

print()
print("FUENTES")

for fuente in resultado["fuentes"]:

    print(
        "-",
        fuente["documento"],
        "| pág.",
        fuente["pagina"],
        "| score",
        round(
            fuente["score"],
            4
        )
    )

RESPUESTA
Según el perfil objetivo de la Ingeniería en Sistemas Computacionales, el perfil de egreso de esta carrera es:

* Implementar aplicaciones computacionales para solucionar problemas de diversos contextos, integrando diferentes tecnologías, plataformas o dispositivos.
* Diseñar, desarrollar y aplicar modelos computacionales para solucionar problemas, mediante la selección y uso de herramientas matemáticas.
* Diseñar e implementar interfaces para la automatización de sistemas de hardware y desarrollo del software asociado.
* Coordinar y participar en equipos multidisciplinarios para la aplicación de soluciones innovadoras en diferentes contextos.
* Diseñar, implementar y administrar bases de datos optimizando los recursos disponibles, conforme a las normas vigentes de manejo y seguridad de la información.
* Desarrollar y administrar software para apoyar la productividad y competitividad de las organizaciones cumpliendo con estándares de calidad.
* Evaluar tecnologías de hardware

In [44]:
resultado = responder_rag(
    "¿Qué información tienes sobre Ingeniería en Sistemas Computacionales?"
)

print(resultado["respuesta"])

La información disponible sobre Ingeniería en Sistemas Computacionales se encuentra en los siguientes documentos:

* Perfil-Objetivo_ISC.pdf: Este documento describe el perfil de egreso de la carrera, que busca formar profesionistas líderes con visión estratégica y amplio sentido ético, capaces de diseñar, desarrollar, implementar y administrar tecnología computacional para aportar soluciones innovadoras en beneficio de la sociedad.
* Reticula_ISC.pdf: Este documento muestra la retícula curricular de la carrera, que incluye asignaturas como Cálculo Diferencial, Cálculo Integral, Cálculo Vectorial, Ecuaciones Diferenciales, Graficación, Lenguajes y Autómatas I y II, Programación Lógica y Funcional, Inteligencia Artificial, y otras relacionadas con la informática.
* Manual_de_Lineamientos_TecNM.pdf: En la página 245, se menciona el ejemplo de diploma para Salida Lateral de Ingeniería en Sistemas Computacionales, que incluye competencias como manejar procesadores de hoja de cálculo digita

In [45]:
resultado = agente(
    "¿Qué información tienes sobre Ingeniería en Sistemas Computacionales?"
)

print(resultado)

{'respuesta': 'Según el perfil objetivo de la Ingeniería en Sistemas Computacionales, se busca formar profesionistas líderes con visión estratégica y amplio sentido ético, capaces de diseñar, desarrollar, implementar y administrar tecnología computacional para aportar soluciones innovadoras en beneficio de la sociedad.', 'estado': 'ok', 'tiempo': 2.71, 'fuentes': [{'chunk_id': 293, 'documento': 'Perfil-Objetivo_ISC.pdf', 'pagina': 1, 'texto': '©TecNM mayo 2016 Página | 1 TECNOLÓGICO NACIONAL DE MÉXICO Secretaría Académica, de Investigación e Innovación Dirección de Docencia e Innovación Educativa Ingeniería en Sistemas Computacionales ISIC-2010-224 Objetivo General Formar profesionistas líderes con visión estratégica y amplio sentido ético; capaz de diseñar, desarrollar, implementar y administrar tecnología computacional para aportar soluciones innovadoras en beneficio de la sociedad; en un contexto global, multidisciplinario y sostenible. Perfil de Egreso 1. Implementa aplicaciones co

# ETAPA 4 — Agente y seguridad

En esta etapa se agregan dos mecanismos de seguridad:

- **Prompt Guard** analiza el mensaje del usuario antes de enviarlo al modelo.
- **Llama Guard** analiza la respuesta generada antes de entregarla.

El flujo será:

Usuario → Prompt Guard → RAG → Llama 3.1 → Llama Guard → Respuesta

In [7]:
# ============================================================
# ETAPA 4 - PROMPT GUARD + LLAMA GUARD
# ============================================================

!pip install -q transformers

# Llama Guard 3 1B en Ollama
!ollama pull llama-guard3:1b

In [8]:
# ============================================================
# SEGURIDAD DEL AGENTE
# ============================================================

from transformers import pipeline

# ------------------------------------------------------------
# PROMPT GUARD
# ------------------------------------------------------------

prompt_guard = pipeline(
    "text-classification",
    model="meta-llama/Llama-Prompt-Guard-2-86M",
    device=-1
)

def revisar_prompt(texto):

    resultado = prompt_guard(
        texto,
        truncation=True,
        max_length=512
    )[0]

    return resultado


# ------------------------------------------------------------
# LLAMA GUARD
# ------------------------------------------------------------

def revisar_respuesta(pregunta, respuesta):

    r = requests.post(
        OLLAMA_URL,
        json={
            "model": "llama-guard3:1b",
            "messages": [
                {
                    "role": "user",
                    "content": pregunta
                },
                {
                    "role": "assistant",
                    "content": respuesta
                }
            ],
            "stream": False
        },
        timeout=120
    )

    r.raise_for_status()

    return r.json()["message"]["content"]


print("✓ Prompt Guard listo")
print("✓ Llama Guard listo")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✓ Prompt Guard listo
✓ Llama Guard listo


In [38]:
# ============================================================
# AGENTE CON SEGURIDAD
# ============================================================

def agente(pregunta):

    # 1. Prompt Guard
    seguridad_prompt = revisar_prompt(
        pregunta
    )

    if seguridad_prompt["label"] != "LABEL_0":
        return {
            "respuesta": "La solicitud fue bloqueada por seguridad.",
            "estado": "prompt_bloqueado"
        }

    # 2. RAG + Llama
    resultado = responder_rag(
        pregunta
    )

    respuesta = resultado["respuesta"]

    # 3. Llama Guard
    seguridad_salida = revisar_respuesta(
        pregunta,
        respuesta
    )

    if seguridad_salida.lower().startswith("unsafe"):
        return {
            "respuesta": "La respuesta fue bloqueada por seguridad.",
            "estado": "salida_bloqueada"
        }

    return {
        "respuesta": respuesta,
        "estado": "ok",
        "tiempo": resultado["tiempo"],
        "fuentes": resultado["fuentes"]
    }

In [10]:
resultado = agente(
    "¿Cuál es el perfil de egreso de Ingeniería en Sistemas Computacionales?"
)

print(resultado["estado"])
print()
print(resultado["respuesta"])

prompt_bloqueado

La solicitud fue bloqueada por seguridad.


# ETAPA 5 — FastAPI y monitoreo básico

En esta etapa se crea una API con FastAPI.

Se agregan dos endpoints:

- `/health` para comprobar que el servicio está activo.
- `/ask` para enviar una pregunta al agente.

También se registra monitoreo básico:

- número de consultas;
- bloqueos por Prompt Guard;
- bloqueos por Llama Guard;
- errores;
- tiempo de respuesta.

In [59]:
# ============================================================
# ETAPA 5 - FASTAPI + MONITOREO BÁSICO
# ============================================================

from fastapi import FastAPI
from pydantic import BaseModel
import time

app = FastAPI(
    title="Agente ISC"
)

metricas = {
    "consultas": 0,
    "prompt_bloqueados": 0,
    "salidas_bloqueadas": 0,
    "errores": 0
}


class Pregunta(BaseModel):
    pregunta: str


@app.get("/health")
def health():

    return {
        "status": "ok",
        "modelo": LLM_MODEL,
        "consultas": metricas["consultas"]
    }


@app.post("/ask")
def ask(request: Pregunta):

    inicio = time.time()

    try:

        metricas["consultas"] += 1

        resultado = agente(
            request.pregunta
        )

        if resultado["estado"] == "prompt_bloqueado":
            metricas["prompt_bloqueados"] += 1

        if resultado["estado"] == "salida_bloqueada":
            metricas["salidas_bloqueadas"] += 1

        return {
            "respuesta": resultado["respuesta"],
            "estado": resultado["estado"],
            "tiempo": round(
                time.time() - inicio,
                2
            )
        }

    except Exception as error:

        metricas["errores"] += 1

        return {
            "estado": "error",
            "respuesta": str(error)
        }

In [64]:
# ============================================================
# INICIAR FASTAPI
# ============================================================

import uvicorn
import threading

def iniciar_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


threading.Thread(
    target=iniciar_api,
    daemon=True
).start()

time.sleep(3)

print("✓ FastAPI iniciado")

INFO:     Started server process [4198]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✓ FastAPI iniciado


In [66]:
respuesta = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "pregunta": (
            "¿Cuál es el perfil de egreso "
            "de Ingeniería en Sistemas Computacionales?"
        )
    },
    timeout=180
)

print(respuesta.json())

WhatsApp: 5215536699933 → 525536699933 | 200
✓ Respuesta enviada
INFO:     2a03:2880:2ff:58:::0 - "POST /webhook HTTP/1.1" 200 OK
Mensaje recibido: 5215536699933 Hola
WhatsApp: 5215536699933 → 525536699933 | 200
✓ Respuesta enviada
INFO:     2a03:2880:10ff:e:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:47:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:9:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:54:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:44:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     127.0.0.1:58700 - "POST /ask HTTP/1.1" 200 OK
{'respuesta': 'El perfil de egreso de la carrera de Ingeniería en Sistemas Computacionales es el siguiente:\n\n* Implementa aplicaciones computacionales para solucionar problemas de diversos contextos, integrando diferentes tecnologías, plataformas o dispositivos.\n* Diseña, desarrolla y aplica modelos computacionales para solucionar problemas, mediante la selección y uso de herram

# ETAPA 6 — WhatsApp + seguimiento de usuarios

En esta etapa se integra WhatsApp Cloud API y una memoria conversacional básica.

Por cada número de WhatsApp se conserva:

- si ya recibió el saludo inicial;
- los últimos mensajes de la conversación;
- si actualmente se está procesando una respuesta;
- mensajes pendientes.

Si un usuario envía varios mensajes mientras el modelo está ocupado,
los mensajes se procesan en orden.


In [63]:
# ============================================================
# ETAPA 6 - WHATSAPP + SEGUIMIENTO DE USUARIOS
# ============================================================

from fastapi import Request, Response, BackgroundTasks
from threading import Lock
import requests

GRAPH_URL = (
    f"https://graph.facebook.com/v21.0/"
    f"{PHONE_NUMBER_ID}/messages"
)

# ------------------------------------------------------------
# MEMORIA TEMPORAL DE USUARIOS
# ------------------------------------------------------------

usuarios = {}
lock_usuarios = Lock()


# ------------------------------------------------------------
# NORMALIZAR NÚMERO
# ------------------------------------------------------------

def normalizar_numero(numero):

    numero = "".join(
        c for c in str(numero)
        if c.isdigit()
    )

    # México: 521XXXXXXXXXX → 52XXXXXXXXXX
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    return numero


# ------------------------------------------------------------
# ENVIAR MENSAJE A WHATSAPP
# ------------------------------------------------------------

def enviar_whatsapp(numero, texto):

    numero_original = numero
    numero = normalizar_numero(numero)

    respuesta = requests.post(
        GRAPH_URL,
        headers={
            "Authorization": f"Bearer {WHATSAPP_TOKEN}",
            "Content-Type": "application/json"
        },
        json={
            "messaging_product": "whatsapp",
            "to": numero,
            "type": "text",
            "text": {
                "body": texto
            }
        },
        timeout=30
    )

    print(
        "WhatsApp:",
        numero_original,
        "→",
        numero,
        "|",
        respuesta.status_code
    )

    if not respuesta.ok:
        print(respuesta.text)

    return respuesta.ok


# ------------------------------------------------------------
# CREAR / OBTENER USUARIO
# ------------------------------------------------------------

def obtener_usuario(numero):

    if numero not in usuarios:

        usuarios[numero] = {
            "saludado": False,
            "procesando": False,
            "pendientes": [],
            "historial": []
        }

    return usuarios[numero]


# ------------------------------------------------------------
# DETECTAR SALUDO
# ------------------------------------------------------------

def es_saludo(texto):

    saludos = {
        "hola",
        "buenos dias",
        "buenos días",
        "buenas tardes",
        "buenas noches",
        "hey",
        "que tal",
        "qué tal"
    }

    texto = texto.lower().strip(" ¿?¡!.,")

    return texto in saludos


# ------------------------------------------------------------
# MENSAJE DE BIENVENIDA
# ------------------------------------------------------------

def mensaje_bienvenida():

    return (
        "Hola 👋 Soy el asistente académico del TESOEM.\n\n"
        "Puedo ayudarte a consultar información sobre:\n"
        "• Ingeniería en Sistemas Computacionales\n"
        "• perfil de egreso\n"
        "• reglamentos\n"
        "• servicio social\n"
        "• lineamientos académicos\n"
        "• retícula y asignaturas\n\n"
        "Escribe tu pregunta y con gusto te ayudo."
    )


# ------------------------------------------------------------
# PROCESAR COLA DE MENSAJES DEL USUARIO
# ------------------------------------------------------------

def procesar_usuario(numero):

    while True:

        with lock_usuarios:

            usuario = obtener_usuario(numero)

            if not usuario["pendientes"]:
                usuario["procesando"] = False
                break

            texto = usuario["pendientes"].pop(0)

            primer_contacto = not usuario["saludado"]

            if primer_contacto:
                usuario["saludado"] = True

            historial = usuario["historial"][-4:]

        # ----------------------------------------------------
        # SALUDO / BIENVENIDA
        # ----------------------------------------------------

        if primer_contacto or es_saludo(texto):

            enviar_whatsapp(
                numero,
                mensaje_bienvenida()
            )

            # Si solo saludó, no enviar al RAG
            if es_saludo(texto):
                continue

        # ----------------------------------------------------
        # CONTEXTO CONVERSACIONAL
        # ----------------------------------------------------

        if historial:

            contexto_previo = "\n".join(
                historial
            )

            pregunta_agente = f"""
Conversación previa:
{contexto_previo}

Mensaje actual:
{texto}
""".strip()

        else:

            pregunta_agente = texto

        # ----------------------------------------------------
        # AGENTE
        # ----------------------------------------------------

        try:

            print()
            print(
                "Procesando:",
                numero,
                "|",
                texto
            )

            resultado = agente(
                pregunta_agente
            )

            respuesta = resultado["respuesta"]

            enviar_whatsapp(
                numero,
                respuesta
            )

            # Guardar historial corto
            with lock_usuarios:

                usuario = obtener_usuario(numero)

                usuario["historial"].extend([
                    f"Usuario: {texto}",
                    f"Asistente: {respuesta}"
                ])

                usuario["historial"] = (
                    usuario["historial"][-6:]
                )

        except Exception as error:

            print(
                "ERROR procesando mensaje:",
                repr(error)
            )

            enviar_whatsapp(
                numero,
                "Ocurrió un error al procesar tu mensaje."
            )


# ------------------------------------------------------------
# RECIBIR MENSAJE Y AGREGAR A COLA
# ------------------------------------------------------------

def recibir_mensaje(numero, texto):

    with lock_usuarios:

        usuario = obtener_usuario(numero)

        usuario["pendientes"].append(
            texto
        )

        # Si ya se está procesando otro mensaje,
        # este queda esperando en la cola
        if usuario["procesando"]:

            print(
                "Mensaje agregado a cola:",
                numero
            )

            return

        usuario["procesando"] = True

    procesar_usuario(
        numero
    )


# ------------------------------------------------------------
# GET - VERIFICAR WEBHOOK
# ------------------------------------------------------------

@app.get("/webhook")
async def verificar_webhook(request: Request):

    params = request.query_params

    if (
        params.get("hub.mode") == "subscribe"
        and
        params.get("hub.verify_token") == VERIFY_TOKEN
    ):

        return Response(
            content=params.get(
                "hub.challenge",
                ""
            ),
            media_type="text/plain"
        )

    return Response(
        content="Token inválido",
        status_code=403
    )


# ------------------------------------------------------------
# POST - RECIBIR MENSAJES
# ------------------------------------------------------------

@app.post("/webhook")
async def webhook(
    request: Request,
    background_tasks: BackgroundTasks
):

    data = await request.json()

    try:

        mensaje = (
            data["entry"][0]
            ["changes"][0]
            ["value"]["messages"][0]
        )

        if mensaje.get("type") != "text":
            return {
                "status": "ignored"
            }

        numero = mensaje["from"]
        texto = mensaje["text"]["body"]

        print()
        print("=" * 60)
        print("MENSAJE RECIBIDO")
        print("=" * 60)

        print("Número:", numero)
        print("Texto:", texto)

        background_tasks.add_task(
            recibir_mensaje,
            numero,
            texto
        )

    except (KeyError, IndexError):

        # Meta también envía eventos de estado:
        # enviado, entregado, leído, etc.
        pass

    return {
        "status": "ok"
    }


print("✓ WhatsApp + seguimiento de usuarios listo")

✓ WhatsApp + seguimiento de usuarios listo


In [58]:
usuarios.clear()

print("✓ Memoria de usuarios reiniciada")

✓ Memoria de usuarios reiniciada


# ETAPA 7 — ngrok y prueba integral

En esta etapa se publica FastAPI mediante un túnel HTTPS de ngrok.

Esto permite que WhatsApp Cloud API pueda acceder al webhook ejecutado en Google Colab.

El flujo final será:

WhatsApp → ngrok → FastAPI → Prompt Guard → RAG → Llama 3.1 → Llama Guard → WhatsApp

In [65]:
# ============================================================
# ETAPA 7 - NGROK Y PRUEBA FINAL
# ============================================================

from pyngrok import ngrok
import requests

# Cerrar túneles anteriores
ngrok.kill()

# Configurar token
ngrok.set_auth_token(
    NGROK_TOKEN
)

# Abrir túnel hacia FastAPI
tunnel = ngrok.connect(
    8000
)

PUBLIC_URL = tunnel.public_url

print("=" * 60)
print("AGENTE ISC PUBLICADO")
print("=" * 60)

print("URL pública:")
print(PUBLIC_URL)

print()
print("Webhook:")
print(PUBLIC_URL + "/webhook")

print()
print("Health:")
print(PUBLIC_URL + "/health")

AGENTE ISC PUBLICADO
URL pública:
https://unneeded-playhouse-tuition.ngrok-free.dev

Webhook:
https://unneeded-playhouse-tuition.ngrok-free.dev/webhook

Health:
https://unneeded-playhouse-tuition.ngrok-free.dev/health


In [35]:
respuesta = requests.get(
    PUBLIC_URL + "/health",
    timeout=30
)

print(
    "Health público:",
    respuesta.status_code
)

print(
    respuesta.json()
)

INFO:     34.169.24.115:0 - "GET /health HTTP/1.1" 200 OK
Health público: 200
{'status': 'ok', 'modelo': 'llama3.1:8b', 'consultas': 2}


In [37]:
print(
    revisar_prompt(
        "Hola, ¿cuál es el perfil de egreso de Ingeniería en Sistemas Computacionales?"
    )
)

{'label': 'LABEL_0', 'score': 0.9995173215866089}
